# Attention-LoRA round on Colab (qwen3-coder / qwen3.6)

Trains the attention LoRA on `lora_dataset.jsonl` and merges to a
q4_k_m GGUF — the same pipeline `run_round4_when_vram_free.sh` ran on
the desktop, moved here so the 30B+ offload never touches local RAM.

**Model:** `MODEL` in the params cell defaults to
`unsloth/Qwen3-Coder-30B-A3B-Instruct`. For a qwen3.6 round set
`AILM_TRAIN_MODEL='unsloth/Qwen3.6-35B-A3B'` in a cell above it — the
checkpoint is a VL model and the loader trains its language stack
text-only.

**Runtime:** GPU required. A100 (40GB) fits the 4-bit model plus
activations; L4 (24GB) works with block-swap into the VM's RAM.
Free-tier T4 (16GB VRAM / ~13GB RAM) cannot hold a 4-bit 30B.

**Data:** `lora_dataset.jsonl` stays private — put it in
`MyDrive/ailm-train/` or upload it when the dataset cell asks.

**Artifacts:** adapter (~53MB) always; merged GGUF (~18.5GB) when
disk allows. Copied to `MyDrive/ailm-train/out/` when Drive is
mounted, else served over a cloudflared quick tunnel whose URL is
posted to the ntfy topic printed by the publish cell.

Back on the machine:

```bash
./fetch_q3c_round.sh              # auto-discovers the URL from ntfy
./fetch_q3c_round.sh --url <URL>  # or pass the printed tunnel URL
# qwen3.6 round registers as its own tag (auto-picks Modelfile.qwen36ft):
./fetch_q3c_round.sh --tag qwen3.6:35b-a3b-ft
```

If artifacts went to Drive, point it at the synced folder instead:
`./fetch_q3c_round.sh --dir ~/path/to/ailm-train/out [--tag ...]`

Leave the keep-alive cell running until the download finishes.

In [ ]:
import shutil, subprocess

out = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True).strip()
name, vram_mb = out.split(', ')
vram_gb = int(vram_mb) / 1024
ram_gb = int(
    l.split()[1] for l in open('/proc/meminfo') if l.startswith('MemAvailable')
) / 1024**2
disk_gb = shutil.disk_usage('/content').free / 1024**3
print(f'GPU: {name}  vram={vram_gb:.0f}GB  ram_avail={ram_gb:.0f}GB  disk_free={disk_gb:.0f}GB')
if vram_gb < 20 and ram_gb < 32:
    raise SystemExit(
        'Not enough memory anywhere for a 4-bit 30B (needs ~18GB weights + '
        'activations). Runtime -> Change runtime type -> A100 or L4.')
if vram_gb < 20:
    print('VRAM under 20GB: most layers will sit in RAM — slow but possible.')
if disk_gb < 100:
    print('Under 100GB free disk: the GGUF merge step will be skipped (adapter still saves).')

In [ ]:
!pip install -q unsloth unsloth_zoo trl datasets hf_transfer
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/bin/cloudflared
!chmod +x /usr/bin/cloudflared
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # 61GB bf16 pull needs it
print('deps ok')

In [ ]:
from pathlib import Path
import os, shutil

DATA = Path('/content/lora_dataset.jsonl')
DRIVE_DIR = Path('/content/drive/MyDrive/ailm-train')
MOUNT_DRIVE = True   # set False to skip the Drive auth click and upload instead

if MOUNT_DRIVE and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

if DATA.exists():
    src = 'already on disk'
elif (DRIVE_DIR / 'lora_dataset.jsonl').exists():
    shutil.copy(DRIVE_DIR / 'lora_dataset.jsonl', DATA)
    src = 'drive'
elif os.environ.get('AILM_DATASET_URL'):
    import urllib.request
    urllib.request.urlretrieve(os.environ['AILM_DATASET_URL'], DATA)
    src = 'url'
else:
    from google.colab import files
    print('upload lora_dataset.jsonl')
    up = files.upload()
    Path(next(iter(up))).rename(DATA)
    src = 'upload'

rows = [l for l in open(DATA, encoding='utf-8') if l.strip()]
print(f'dataset: {src} -> {DATA}  rows={len(rows)}  size={DATA.stat().st_size/1e6:.1f}MB')

In [ ]:
import os
from pathlib import Path

# Ungated bf16 source, loaded 4-bit. Qwen3.6 round: set
# AILM_TRAIN_MODEL='unsloth/Qwen3.6-35B-A3B' in a cell above this one
# (it is a VL checkpoint - the loader below strips the vision tower).
MODEL = (os.environ.get('AILM_TRAIN_MODEL')
         or 'unsloth/Qwen3-Coder-30B-A3B-Instruct')
OUT = Path('/content/out')
LORA_RANK = 16
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
NTFY = 'ailm-train-q3c-9x4k2mqwz7f3'   # artifact channel — NOT the shim's ailm-ep topic
DO_GGUF = 'auto'                     # auto | always | never
HF_PUSH_REPO = os.environ.get('AILM_HF_REPO')  # optional 'user/repo' for hub push
# os.environ['HF_TOKEN'] = 'hf_...'  # only needed for HF_PUSH_REPO
OUT.mkdir(exist_ok=True)
print('model ->', MODEL)
print('out ->', OUT)

In [ ]:
import gc, json
import torch
from unsloth import FastLanguageModel
from unsloth_zoo.block_swap import estimate_training_reserve_bytes, find_decoder_layers
from unsloth_zoo.device_map_planner import build_meta_model
from transformers import AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# Empirical seq_len from the real dataset — no arbitrary truncation.
_t = AutoTokenizer.from_pretrained(MODEL)
_obs = max(
    len(_t.encode(r['prompt'] + r['completion']))
    for r in (json.loads(l) for l in open(DATA, encoding='utf-8') if l.strip())
)
MAX_SEQ = ((_obs + 63) // 64) * 64

# Empirical attention-LoRA reserve so the planner doesn't charge for
# unused MoE expert adapters.
_meta, _hf_q, _mcfg = build_meta_model(MODEL, load_in_4bit=True)
_layers = find_decoder_layers(_meta)
_attn = [m for l in _layers for n, m in l.named_modules()
         if n.endswith(tuple(TARGET_MODULES))]
_lora_b = sum((m.in_features + m.out_features) * LORA_RANK for m in _attn) * 20
RESERVE = int(estimate_training_reserve_bytes(_mcfg, MAX_SEQ, 1, extra_bytes=_lora_b))
# VL checkpoints (qwen3.6 is Qwen3_5MoeForConditionalGeneration) carry a
# vision tower; text_only loads the language stack alone, which is all
# this dataset trains. GGUF export then emits the text model.
_is_vl = bool(getattr(_mcfg, 'vision_config', None))
_load_kw = {'text_only': True} if _is_vl else {}
del _meta, _layers, _attn, _t
gc.collect()
print(f'max_tokens={_obs} seq_len={MAX_SEQ} reserve={RESERVE/2**30:.1f}GiB vl={_is_vl}')

# Offload budget derived from THIS vm's memory — never a fixed guess.
_free_vram = torch.cuda.mem_get_info(0)[0]
_free_ram = int(
    l.split()[1] for l in open('/proc/meminfo') if l.startswith('MemAvailable')
) * 1024
max_memory = {0: int(_free_vram * 0.92), 'cpu': int(_free_ram * 0.5)}
print(f'max_memory: gpu={max_memory[0]/2**30:.1f}GiB cpu={max_memory["cpu"]/2**30:.1f}GiB')

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    offload_layers='auto',
    max_memory=max_memory,
    device_map_planner_kwargs={'reserve_bytes': RESERVE},
    **_load_kw,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, target_modules=TARGET_MODULES,
    lora_alpha=32, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth')

ds = load_dataset('json', data_files=str(DATA), split='train')
print('rows:', len(ds))
cfg = SFTConfig(
    output_dir=str(OUT / 'adapter'),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=3,
    logging_steps=1,
    save_strategy='no',
    bf16=True,
    max_length=MAX_SEQ,
    completion_only_loss=True,
    optim='adamw_8bit',
    report_to=[],
    seed=42,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok)
trainer.train()
model.save_pretrained(OUT / 'adapter')
tok.save_pretrained(OUT / 'adapter')
print('saved adapter ->', OUT / 'adapter')

In [ ]:
import shutil
from pathlib import Path

MERGED = OUT / 'merged'
_disk_gb = shutil.disk_usage('/content').free / 1024**3
_do = DO_GGUF == 'always' or (DO_GGUF == 'auto' and _disk_gb >= 100)
if _do:
    # save_pretrained_gguf writes ~60GB of bf16 shards, then quantizes.
    model.save_pretrained_gguf(str(MERGED), tok, quantization_method='q4_k_m')
    print('merged ->', MERGED)
else:
    print(f'GGUF merge skipped (disk_free={_disk_gb:.0f}GB, DO_GGUF={DO_GGUF}). '
          'Adapter still saved — merge on the desktop or rerun with more disk.')

In [ ]:
import hashlib, json, shutil, tarfile, urllib.request
from pathlib import Path

# adapter tarball so the whole round travels as one file
_atar = OUT / 'adapter.tar.gz'
with tarfile.open(_atar, 'w:gz') as t:
    t.add(OUT / 'adapter', arcname='adapter')

def _sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()

files = [_atar] + sorted((MERGED).glob('**/*.gguf')) if MERGED.exists() else [_atar]
manifest = [{'name': p.name, 'size': p.stat().st_size, 'sha256': _sha(p)} for p in files]
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=1))
print(json.dumps(manifest, indent=1))

# optional: private HF repo
if HF_PUSH_REPO:
    from huggingface_hub import upload_file
    for m in manifest:
        upload_file(path_or_fileobj=str(OUT / m['name']) if (OUT / m['name']).exists()
                    else str(next(MERGED.glob('**/' + m['name']))),
                    path_in_repo=m['name'], repo_id=HF_PUSH_REPO)
    upload_file(path_or_fileobj=str(OUT / 'manifest.json'),
                path_in_repo='manifest.json', repo_id=HF_PUSH_REPO)
    print('pushed ->', HF_PUSH_REPO)

# Drive copy when mounted
_drive_out = Path('/content/drive/MyDrive/ailm-train/out')
if Path('/content/drive/MyDrive').exists():
    _drive_out.mkdir(parents=True, exist_ok=True)
    for m in manifest + [{'name': 'manifest.json'}]:
        src = OUT / m['name']
        if not src.exists():
            src = next(MERGED.glob('**/' + m['name']))
        shutil.copy(src, _drive_out / m['name'])
        print('drive:', _drive_out / m['name'])
    print('artifacts in Drive — run fetch_q3c_round.sh --dir on the synced folder')
else:
    print('no Drive mount — next cell serves OUT over a cloudflared tunnel')

In [ ]:
import re, subprocess, threading, time, urllib.request
from pathlib import Path

if Path('/content/drive/MyDrive').exists():
    print('Drive has the artifacts — tunnel not needed, stop here.')
else:
    import http.server, functools
    def _serve():
        handler = functools.partial(http.server.SimpleHTTPRequestHandler,
                                    directory=str(OUT))
        http.server.ThreadingHTTPServer(('127.0.0.1', 8787), handler).serve_forever()
    threading.Thread(target=_serve, daemon=True).start()
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787',
         '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in proc.stdout:
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break
    assert url, 'cloudflared gave no URL — check output above'
    print('artifact URL:', url)
    print('manifest   :', url + '/manifest.json')
    # publish to the TRAIN topic — not the shim's ailm-ep endpoint topic
    urllib.request.urlopen(
        urllib.request.Request('https://ntfy.sh/' + NTFY, data=url.encode()))
    print('posted to ntfy topic', NTFY)
    print('on the machine: ./fetch_q3c_round.sh')

In [ ]:
import time
print('keep-alive — leave running until fetch_q3c_round.sh finishes')
while True:
    time.sleep(60)